# Notebook Training Model — CDSS DBD (Klasifikasi ICD-10 A90 vs A91)

**Judul Skripsi:** Sistem Pendukung Keputusan Klinis Klasifikasi Penyakit Demam Berdarah Dengue  
Menggunakan Algoritma *Decision Tree* Berbasis Kode ICD-10 di RS Aulia

**Dataset:** Data Rekam Medis Laboratorium RS Aulia  
**Algoritma:** Decision Tree + SMOTE + GridSearchCV  
**Framework:** Scikit-learn, Imbalanced-learn, Python 3

---
### Alur Pemodelan (CRISP-DM):
1. Import Library
2. Muat Dataset
3. Parsing & Preprocessing
4. Eksplorasi Data (EDA)
5. Train-Test Split & SMOTE
6. Skenario 1 — Baseline Model (scoring: recall_macro)
7. Skenario 2 — Model Teroptimasi / Final (scoring: recall)
8. Pemilihan Model Final
9. Evaluasi Lengkap (Confusion Matrix, Classification Report, Feature Importance)
10. Export Pipeline & Metrik

## 1. Import Library

In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
RANDOM_STATE = 42

print("Semua library berhasil diimport.")

## 2. Memuat Dataset

In [ ]:
# Sesuaikan path dataset jika diperlukan
DATASET_PATH = "dataset/Data_Lab_Penyakit_DBD_RS_Aulia.xlsx"

df_raw = pd.read_excel(DATASET_PATH)
print(f"Dimensi dataset: {df_raw.shape[0]} baris x {df_raw.shape[1]} kolom")
df_raw.head(5)

## 3. Parsing & Preprocessing

Data dari Excel mengandung teks gabungan (contoh: `137000 (150,000-450,000)`), sehingga perlu di-parsing untuk mengekstrak nilai numerik saja.

In [ ]:
def parse_usia(teks):
    """Ekstrak tahun dari teks seperti '25 Th 8 Bl 9 Hr'"""
    if pd.isna(teks): return np.nan
    teks = str(teks).strip()
    cocok = re.search(r'(\d+)\s*(Th|tahun)', teks, re.IGNORECASE)
    if cocok: return float(cocok.group(1))
    cocok = re.search(r'(\d+)\s*(Bl|bulan)', teks, re.IGNORECASE)
    if cocok: return 0.0
    return np.nan

def parse_nilai_lab(teks):
    """Ekstrak angka pertama dari teks seperti '137000 (150,000-450,000)'"""
    if pd.isna(teks): return np.nan
    teks = str(teks).strip()
    cocok = re.search(r'([\d]+\.?[\d]*)', teks)
    return float(cocok.group(1)) if cocok else np.nan

# Rename kolom
kolom_dipilih = {
    'Usia (tahun)': 'Usia',
    'Jenis Kelamin\n(L/P)': 'Jenis_Kelamin',
    'Trombosit\n(ribu/\u03bcL)': 'Trombosit',
    'Hematokrit (%)': 'Hematokrit',
    'Hemoglobin (g/dL)': 'Hemoglobin',
    'Leukosit\n(ribu/\u03bcL)': 'Leukosit',
    'kode ICD': 'ICD'
}
df = df_raw[list(kolom_dipilih.keys())].rename(columns=kolom_dipilih).copy()

# Parsing
df['Usia'] = df['Usia'].apply(parse_usia)
for kolom in ['Trombosit', 'Hematokrit', 'Hemoglobin', 'Leukosit']:
    df[kolom] = df[kolom].apply(parse_nilai_lab)

df['Jenis_Kelamin'] = df['Jenis_Kelamin'].astype(str).str.strip().str.upper()
df['ICD'] = df['ICD'].astype(str).str.strip().str.upper()

# Imputasi
kolom_lab = ['Trombosit', 'Hematokrit', 'Hemoglobin', 'Leukosit']
kolom_numerik = ['Usia', 'Trombosit', 'Hematokrit', 'Hemoglobin', 'Leukosit']
kolom_kategorikal = ['Jenis_Kelamin']

imputer_numerik = SimpleImputer(strategy='median')
imputer_kategorikal = SimpleImputer(strategy='most_frequent')

df[kolom_numerik] = imputer_numerik.fit_transform(df[kolom_numerik])
df[kolom_kategorikal] = imputer_kategorikal.fit_transform(df[kolom_kategorikal])

# Hapus duplikat
df = df.drop_duplicates(subset=['Usia','Jenis_Kelamin','Trombosit','Hematokrit','Hemoglobin','Leukosit','ICD']).reset_index(drop=True)

# Standardisasi & Encoding
df['Jenis_Kelamin'] = df['Jenis_Kelamin'].replace({
    'LAKI-LAKI': 'L', 'LAKI LAKI': 'L', 'PRIA': 'L', 'M': 'L',
    'PEREMPUAN': 'P', 'WANITA': 'P', 'F': 'P'
})
df = df[df['Jenis_Kelamin'].isin(['L', 'P'])].reset_index(drop=True)
df['Jenis_Kelamin'] = df['Jenis_Kelamin'].map({'L': 1, 'P': 0})
df['ICD'] = df['ICD'].map({'A91': 1, 'A90': 0})

print(f"Dataset bersih: {df.shape[0]} baris")
print(f"\nDistribusi kelas:")
print(df['ICD'].value_counts().rename({0: 'A90', 1: 'A91'}))
df.head()

## 4. Eksplorasi Data (EDA)

In [ ]:
# Distribusi kelas (class imbalance visualization)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart distribusi kelas
kelas_count = df['ICD'].value_counts().rename({0: 'A90 (Dengue Fever)', 1: 'A91 (DHF)'})
axes[0].pie(kelas_count, labels=kelas_count.index, autopct='%1.1f%%',
            colors=['#4e9af1', '#f76e6e'], startangle=90,
            textprops={'fontsize': 12})
axes[0].set_title('Distribusi Kelas Dataset', fontsize=13, fontweight='bold')

# Boxplot Trombosit per kelas
df_plot = df.copy()
df_plot['Diagnosis'] = df_plot['ICD'].map({0: 'A90', 1: 'A91'})
axes[1].boxplot(
    [df_plot[df_plot['Diagnosis']=='A90']['Trombosit'],
     df_plot[df_plot['Diagnosis']=='A91']['Trombosit']],
    labels=['A90 (Dengue Fever)', 'A91 (DHF)'],
    patch_artist=True,
    boxprops=dict(facecolor='#c6dcfc'),
    medianprops=dict(color='#1a1a2e', linewidth=2)
)
axes[1].set_title('Distribusi Trombosit per Kelas', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Nilai Trombosit (/uL)', fontsize=11)
axes[1].yaxis.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 5. Train-Test Split & SMOTE

In [ ]:
X = df[['Usia', 'Trombosit', 'Hematokrit', 'Hemoglobin', 'Leukosit', 'Jenis_Kelamin']]
y = df['ICD'].astype(int)
FEATURE_NAMES = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"Data Train: {X_train.shape[0]} baris | Data Test: {X_test.shape[0]} baris")

# SMOTE untuk menangani class imbalance
smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print(f"Setelah SMOTE: {y_train_smote.value_counts().rename({0:'A90', 1:'A91'}).to_dict()}")

## 6. Skenario 1 — Baseline Model

**Parameter:**
- `scoring = 'recall_macro'` (rata-rata A90 dan A91)
- `max_depth` mencakup nilai 3 (pohon sangat dangkal)

In [ ]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

param_grid_s1 = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3, 5, 7, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4, 8],
    'class_weight': ['balanced', None]
}
gs_s1 = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_grid=param_grid_s1,
    scoring='recall_macro',
    cv=cv_strategy, n_jobs=-1, verbose=0
)
gs_s1.fit(X_train, y_train)
model_s1 = gs_s1.best_estimator_
y_pred_s1 = model_s1.predict(X_test)

print("=== SKENARIO 1 (BASELINE) ===")
print(f"Best Params : {gs_s1.best_params_}")
print(f"Accuracy    : {accuracy_score(y_test, y_pred_s1)*100:.2f}%")
print(f"Recall A90  : {recall_score(y_test, y_pred_s1, pos_label=0)*100:.2f}%")
print(f"Recall A91  : {recall_score(y_test, y_pred_s1, pos_label=1)*100:.2f}%")
print()
print(classification_report(y_test, y_pred_s1, target_names=['A90', 'A91']))

## 7. Skenario 2 — Model Teroptimasi / Final

**Perubahan dari Skenario 1:**
- `scoring = 'recall'` → fokus memaksimalkan recall kelas A91
- Hapus `max_depth=3` → pohon boleh tumbuh lebih dalam

In [ ]:
param_grid_s2 = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [5, 7, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4, 8],
    'class_weight': ['balanced', None]
}
gs_s2 = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_grid=param_grid_s2,
    scoring='recall',
    cv=cv_strategy, n_jobs=-1, verbose=0
)
gs_s2.fit(X_train, y_train)
model_s2 = gs_s2.best_estimator_
y_pred_s2 = model_s2.predict(X_test)

print("=== SKENARIO 2 (FINAL — TEROPTIMASI) ===")
print(f"Best Params : {gs_s2.best_params_}")
print(f"Accuracy    : {accuracy_score(y_test, y_pred_s2)*100:.2f}%")
print(f"Recall A90  : {recall_score(y_test, y_pred_s2, pos_label=0)*100:.2f}%")
print(f"Recall A91  : {recall_score(y_test, y_pred_s2, pos_label=1)*100:.2f}%")
print()
print(classification_report(y_test, y_pred_s2, target_names=['A90', 'A91']))

## 8. Perbandingan Skenario 1 vs Skenario 2

In [ ]:
# Tabel perbandingan
data_perbandingan = {
    'Metrik': ['Accuracy', 'Precision A90', 'Recall A90', 'F1 A90',
               'Precision A91', 'Recall A91', 'F1 A91'],
    'Skenario 1 (Baseline)': [
        f"{accuracy_score(y_test, y_pred_s1)*100:.1f}%",
        f"{precision_score(y_test, y_pred_s1, pos_label=0)*100:.1f}%",
        f"{recall_score(y_test, y_pred_s1, pos_label=0)*100:.1f}%",
        f"{f1_score(y_test, y_pred_s1, pos_label=0)*100:.1f}%",
        f"{precision_score(y_test, y_pred_s1, pos_label=1)*100:.1f}%",
        f"{recall_score(y_test, y_pred_s1, pos_label=1)*100:.1f}%",
        f"{f1_score(y_test, y_pred_s1, pos_label=1)*100:.1f}%",
    ],
    'Skenario 2 (Final)': [
        f"{accuracy_score(y_test, y_pred_s2)*100:.1f}%",
        f"{precision_score(y_test, y_pred_s2, pos_label=0)*100:.1f}%",
        f"{recall_score(y_test, y_pred_s2, pos_label=0)*100:.1f}%",
        f"{f1_score(y_test, y_pred_s2, pos_label=0)*100:.1f}%",
        f"{precision_score(y_test, y_pred_s2, pos_label=1)*100:.1f}%",
        f"{recall_score(y_test, y_pred_s2, pos_label=1)*100:.1f}%",
        f"{f1_score(y_test, y_pred_s2, pos_label=1)*100:.1f}%",
    ]
}
df_perbandingan = pd.DataFrame(data_perbandingan)
print(df_perbandingan.to_string(index=False))

## 9. Evaluasi Model Final — Confusion Matrix & Feature Importance

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred_s2)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Confusion Matrix Heatmap
labels = [['TN='+str(cm[0,0])+'\n(A90->A90)', 'FP='+str(cm[0,1])+'\n(A90->A91)'],
          ['FN='+str(cm[1,0])+'\n(A91->A90)', 'TP='+str(cm[1,1])+'\n(A91->A91)']]
sns.heatmap(cm, annot=False, cmap='Blues', fmt='d',
            xticklabels=['A90','A91'], yticklabels=['A90','A91'],
            linewidths=2, linecolor='white', ax=axes[0], vmin=0, vmax=350)
for i in range(2):
    for j in range(2):
        clr = 'white' if cm[i,j] > 150 else '#1a1a2e'
        axes[0].text(j+0.5, i+0.5, f'{cm[i,j]}\n{labels[i][j]}',
                     ha='center', va='center', fontsize=11, fontweight='bold', color=clr)
axes[0].set_xlabel('Prediksi Model', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Label Aktual', fontsize=12, fontweight='bold')
axes[0].set_title('Confusion Matrix — Model Final (Skenario 2)', fontsize=13, fontweight='bold')

# ── Feature Importance ────────────────────────────────────────────────────
fi = model_s2.feature_importances_
fi_idx = np.argsort(fi)
colors = ['#f76e6e' if v == fi.max() else '#4e9af1' for v in fi[fi_idx]]
axes[1].barh([FEATURE_NAMES[i] for i in fi_idx], fi[fi_idx]*100,
             color=colors, edgecolor='white', height=0.6)
for i, (v, idx) in enumerate(zip(fi[fi_idx]*100, fi_idx)):
    axes[1].text(v+0.3, i, f'{v:.2f}%', va='center', fontsize=10, fontweight='bold')
axes[1].set_xlabel('Tingkat Kepentingan (%)', fontsize=12, fontweight='bold')
axes[1].set_title('Feature Importance — Model Final', fontsize=13, fontweight='bold')
axes[1].xaxis.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('assets/thesis_figures/gambar_cm_fi_final.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 10. Visualisasi Pohon Keputusan (Decision Tree)

In [ ]:
fig, ax = plt.subplots(figsize=(22, 9))
plot_tree(
    model_s2,
    feature_names=FEATURE_NAMES,
    class_names=['A90', 'A91'],
    filled=True, rounded=True, fontsize=9,
    ax=ax, max_depth=4, impurity=False, proportion=False
)
ax.set_title('Visualisasi Pohon Keputusan — Model Final (Kedalaman 1-4)',
             fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('assets/thesis_figures/gambar_decision_tree_nb.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print("\nAturan Pohon (Teks):")
print(export_text(model_s2, feature_names=FEATURE_NAMES, max_depth=4))

## 11. Export Pipeline & Metrik ke File .pkl

Pipeline yang diekspor sudah termasuk *preprocessor* (ColumnTransformer + SimpleImputer) sehingga aplikasi Streamlit tidak perlu melakukan preprocessing ulang.

In [ ]:
# Build Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('imputer_numerik', SimpleImputer(strategy='median'),
         ['Usia', 'Trombosit', 'Hematokrit', 'Hemoglobin', 'Leukosit']),
        ('imputer_kategorikal', SimpleImputer(strategy='most_frequent'), ['Jenis_Kelamin'])
    ],
    remainder='drop'
)
pipeline_final = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('classifier', clone(model_s2))
])
pipeline_final.fit(X_train, y_train)

# Hitung metrik final
y_pred_final = pipeline_final.predict(X_test)
precision_pc = precision_score(y_test, y_pred_final, average=None, zero_division=0)
recall_pc    = recall_score(y_test, y_pred_final, average=None, zero_division=0)
f1_pc        = f1_score(y_test, y_pred_final, average=None, zero_division=0)
cm_final     = confusion_matrix(y_test, y_pred_final)
cr           = classification_report(y_test, y_pred_final, target_names=['A90','A91'], output_dict=True)
cr_text      = classification_report(y_test, y_pred_final, target_names=['A90','A91'])

dt_final = pipeline_final.named_steps['classifier']

metrics = {
    'accuracy'    : float(accuracy_score(y_test, y_pred_final)),
    'precision'   : float(precision_score(y_test, y_pred_final, average='macro', zero_division=0)),
    'recall'      : float(recall_score(y_test, y_pred_final, average='macro', zero_division=0)),
    'f1_score'    : float(f1_score(y_test, y_pred_final, average='macro', zero_division=0)),
    'train_accuracy': float(accuracy_score(y_train, pipeline_final.predict(X_train))),
    'precision_a90': float(precision_pc[0]), 'recall_a90': float(recall_pc[0]), 'f1_a90': float(f1_pc[0]),
    'precision_a91': float(precision_pc[1]), 'recall_a91': float(recall_pc[1]), 'f1_a91': float(f1_pc[1]),
    'confusion_matrix': cm_final.tolist(),
    'classification_report': cr,
    'classification_report_text': cr_text,
    'feature_importances': dt_final.feature_importances_.tolist(),
    'feature_names': FEATURE_NAMES,
    'class_counts': {'A90': int((y_test==0).sum()), 'A91': int((y_test==1).sum())},
    'n_train': len(X_train), 'n_test': len(X_test), 'n_total': len(df),
    'best_model_name': "Skenario 2 — class_weight balanced, recall scoring"
}

os.makedirs('models', exist_ok=True)
joblib.dump(pipeline_final, 'models/pipeline_dbd.pkl')
joblib.dump(metrics, 'models/eval_metrics.pkl')

print("Pipeline berhasil disimpan ke: models/pipeline_dbd.pkl")
print("Metrik berhasil disimpan ke  : models/eval_metrics.pkl")
print(f"Accuracy : {metrics['accuracy']*100:.2f}%")
print(f"Recall A91: {metrics['recall_a91']*100:.2f}%")
print(cr_text)